In [ ]:
!pip install transformers datasets evaluate accelerate
!pip install -q peft "torchao>=0.16.0"
!pip install minigrid
!pip install stable-baselines3[extra]

In [ ]:
import minigrid
from minigrid.wrappers import ImgObsWrapper
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
import gymnasium as gym
from minigrid.wrappers import ImgObsWrapper
import torch as th
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
import os
from stable_baselines3.common.evaluation import evaluate_policy
import numpy as np
import seaborn as sns
from stable_baselines3.common.monitor import Monitor
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import evaluate
import re
import shutil
import time
from google.colab import files
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
import pandas as pd
from datasets import Dataset

data_path = '/content/enriched_reward_dataset.csv'

try:
    df = pd.read_csv(data_path)
    print("Original Dataset Shape:", df.shape)

    if 'reward' in df.columns and 'label' not in df.columns:
        df = df.rename(columns={'reward': 'label'})

    print("\n--- Class Distribution BEFORE Sampling ---")
    display(df['label'].value_counts().sort_index())

    print("\nPerforming balanced downsampling...")
    min_class_size = df['label'].value_counts().min()
    df_sampled = df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min_class_size, random_state=42))
    # Shuffle the final sampled dataset
    df_sampled = df_sampled.sample(frac=1, random_state=42).reset_index(drop=True)

    print("\n--- Class Distribution AFTER Sampling ---")
    display(df_sampled['label'].value_counts().sort_index())
    print("New Dataset Shape:", df_sampled.shape)
    print("New unique labels in the dataset:", sorted(df_sampled['label'].unique()))

    dataset = Dataset.from_pandas(df_sampled)
except Exception as e:
    print(f"Error loading dataset: {e}")

In [ ]:
import random

if 'dataset' in locals():
    print("Sample of text inputs and their labels:")
    print("-" * 50)
    # Select 5 random samples
    sample_indices = random.sample(range(len(dataset)), 5)
    for idx in sample_indices:
        sample = dataset[idx]
        print(f"TEXT:  {sample.get('text', 'No text column')}")
        print(f"LABEL: {sample.get('label', 'No label column')}")
        print("-" * 50)
else:
    print("Dataset not loaded yet.")

In [ ]:
%load_ext tensorboard
# Launch TensorBoard monitoring both DeBERTa and PPO log directories
# Hugging Face Trainer defaults to logging in <output_dir>/runs
%tensorboard --logdir_spec Deberta:./reward-model-lora/runs,PPO:./ppo_logs/

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
print("CUDA_LAUNCH_BLOCKING enabled. PyTorch will now run synchronously to report exact error locations.")

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-small"
MAX_LENGTH = 256
BATCH_SIZE = 32  # for L4 GPU
EPOCHS = 5
LEARNING_RATE = 2e-4  # LoRA typically needs a learning rate between 1e-4 and 5e-4

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if 'dataset' in locals():
    if 'text' not in dataset.column_names:
        print("Please update the column name in tokenize_function to match your dataset.")
    else:
        unique_rewards = sorted(list(set(dataset["label"])))
        NUM_CLASSES = len(unique_rewards)
        label2id = {str(reward): i for i, reward in enumerate(unique_rewards)}
        id2label = {i: str(reward) for i, reward in enumerate(unique_rewards)}
        print(f"Found {NUM_CLASSES} classes. Mapping: {label2id}")

        def tokenize_function(examples):
            tokenized = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)
            if "label" in examples:
                tokenized["labels"] = [int(label2id[str(float(l))]) for l in examples["label"]]
            return tokenized

        print("Tokenizing dataset...")
        tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)
        tokenized_datasets = tokenized_datasets.train_test_split(test_size=0.2, seed=42)

        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=NUM_CLASSES,
            label2id=label2id,
            id2label=id2label,
            ignore_mismatched_sizes=True
        )

        peft_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            inference_mode=False,
            r=8,
            lora_alpha=16,
            lora_dropout=0.1
        )
        model = get_peft_model(model, peft_config)
        model.print_trainable_parameters()

        metric_acc = evaluate.load("accuracy")
        metric_f1 = evaluate.load("f1")

        def compute_metrics(eval_pred):
            predictions, labels = eval_pred
            preds = np.argmax(predictions, axis=1)
            acc = metric_acc.compute(predictions=preds, references=labels)["accuracy"]
            f1 = metric_f1.compute(predictions=preds, references=labels, average="macro")["f1"]
            return {"accuracy": acc, "f1": f1}

        training_args = TrainingArguments(
            output_dir="./reward-model-lora",
            learning_rate=LEARNING_RATE,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE,
            num_train_epochs=EPOCHS,
            weight_decay=0.01,
            eval_strategy="epoch",
            save_strategy="epoch",
            logging_steps=10,
            report_to="tensorboard",
            load_best_model_at_end=True,
            metric_for_best_model="eval_f1",
            greater_is_better=True,
            save_total_limit=2,
            warmup_ratio=0.1,
            max_grad_norm=1.0,
            bf16=True, # L4 GPUs support bfloat16 which prevents NaN loss
            fp16=False
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_datasets["train"],
            eval_dataset=tokenized_datasets["test"],
            processing_class=tokenizer,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
        )

        print("Starting LoRA fine-tuning with bfloat16")
        start_time = time.time()
        trainer.train()
        end_time = time.time()

        print(f"\nFine-tuning complete in {end_time - start_time:.2f} seconds!\n")

        trainer.save_model("./reward-model-lora")
        print("Training complete and model saved!")

        print("Zipping the model for download")
        shutil.make_archive("reward-model-lora", 'zip', "./reward-model-lora")
        print("Downloading the model")
        files.download("reward-model-lora.zip")


In [ ]:
if 'tokenized_datasets' in locals():
    unique_train_labels = set(tokenized_datasets["train"]["labels"])
    print("Unique training labels:", unique_train_labels)

    if all(isinstance(l, int) and 0 <= l < NUM_CLASSES for l in unique_train_labels):
        print("Label mapping is correct! All labels are valid integers within the expected range.")
    else:
        print("ERROR: Invalid labels found! They must be integers from 0 to", NUM_CLASSES - 1)
else:
    print("tokenized_datasets not found. Please run the tokenization cell first.")

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

if 'tokenized_datasets' in locals() and 'id2label' in locals():
    train_labels = tokenized_datasets["train"]["labels"]
    label_counts = Counter(train_labels)

    reward_labels = {id2label[label_id]: count for label_id, count in label_counts.items()}

    print("Class distribution in training set:")
    for label, count in sorted(reward_labels.items()):
        print(f"Reward {label}: {count} samples")

    # Plotting the class distribution
    plt.figure(figsize=(10, 6))
    sns.barplot(x=list(reward_labels.keys()), y=list(reward_labels.values()), palette='viridis')
    plt.title('Distribution of Reward Classes in Training Set')
    plt.xlabel('Reward Value')
    plt.ylabel('Number of Samples')
    plt.xticks(rotation=45)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("tokenized_datasets or id2label not found. Please run the tokenization and model setup cells first.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("Evaluating the model on the test set...")
eval_results = trainer.evaluate()
print("\nOverall Evaluation results:")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

print("\nGenerating predictions on the test set...")
predictions = trainer.predict(tokenized_datasets["test"])
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

id2label = trainer.model.config.id2label

print("\nSample Predictions vs Actual Labels:")
print("-" * 40)
for i in range(10):
    pred_label = id2label[preds[i]] if preds[i] in id2label else preds[i]
    actual_label = id2label[labels[i]] if labels[i] in id2label else labels[i]
    print(f"Predicted: {pred_label} | Actual: {actual_label}")

print("\nPlotting training history...")
log_history = trainer.state.log_history

train_loss = [log['loss'] for log in log_history if 'loss' in log]
train_steps = [log['step'] for log in log_history if 'loss' in log]

eval_loss = [log['eval_loss'] for log in log_history if 'eval_loss' in log]
eval_steps = [log['step'] for log in log_history if 'eval_loss' in log]

plt.figure(figsize=(10, 5))
if train_loss:
    plt.plot(train_steps, train_loss, label='Training Loss')
if eval_loss:
    plt.plot(eval_steps, eval_loss, label='Validation Loss', marker='o')

plt.title('DeBERTa Fine-tuning Loss (Cross-Entropy)')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

print("\nPlotting Accuracy and F1 Score...")
eval_acc = [log['eval_accuracy'] for log in log_history if 'eval_accuracy' in log]
eval_f1 = [log['eval_f1'] for log in log_history if 'eval_f1' in log]
eval_metric_steps = [log['step'] for log in log_history if 'eval_accuracy' in log]

if eval_acc and eval_f1:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    ax1.plot(eval_metric_steps, eval_acc, label='Validation Accuracy', marker='o', color='green')
    ax1.set_title('DeBERTa Fine-tuning Accuracy')
    ax1.set_xlabel('Steps')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(eval_metric_steps, eval_f1, label='Validation F1 Score', marker='o', color='purple')
    ax2.set_title('DeBERTa Fine-tuning F1 Score')
    ax2.set_xlabel('Steps')
    ax2.set_ylabel('F1 Score')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()
else:
    print("Accuracy and F1 metrics not found in log history. Ensure evaluation includes them.")

print("\nPlotting Confusion Matrix...")
cm = confusion_matrix(labels, preds)
display_labels = [id2label[i] for i in sorted(id2label.keys())] if id2label else None

fig, ax = plt.subplots(figsize=(8, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_labels)
disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation=45)
plt.title('Confusion Matrix: Predicted vs Actual Rewards')
plt.tight_layout()
plt.show()


In [ ]:
import gymnasium as gym
from minigrid.wrappers import ImgObsWrapper
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from peft import PeftModel

class DebertaRewardWrapper(gym.Wrapper):
    def __init__(self, env, reward_model_path="./reward-model-lora"):
        super().__init__(env)
        self.action_map = {0: 'turn left', 1: 'turn right', 2: 'move forward', 3: 'pick up', 4: 'drop', 5: 'toggle or open', 6: 'done'}
        self.dir_map = {0: 'right', 1: 'down', 2: 'left', 3: 'up'}
        self.visited_positions = set()
        self.has_picked_up_key = False
        self.has_opened_door = False
        self.has_reached_goal = False

        try:
            print(f"Attempting to load PEFT model from {reward_model_path}...")

            label2id = {'-3.5': 0, '-2.5': 1, '-0.1': 2, '2.0': 3, '3.0': 4, '10.0': 5}
            id2label = {0: '-3.5', 1: '-2.5', 2: '-0.1', 3: '2.0', 4: '3.0', 5: '10.0'}

            tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")

            base_model = AutoModelForSequenceClassification.from_pretrained(
                "microsoft/deberta-v3-small",
                num_labels=len(label2id),
                id2label=id2label,
                label2id=label2id,
                ignore_mismatched_sizes=True
            )

            peft_model = PeftModel.from_pretrained(base_model, reward_model_path)

            device = 0 if torch.cuda.is_available() else -1
            self.reward_pipeline = pipeline("text-classification", model=peft_model, tokenizer=tokenizer, device=device)
            print(f"Successfully loaded fine-tuned LoRA model from {reward_model_path}")

        except Exception as e:
            print(f"Could not load fine-tuned model, using base model. Error: {e}")
            self.reward_pipeline = pipeline("text-classification", model="microsoft/deberta-v3-small")

    def get_text_description(self, action):
        unwrapped = self.env.unwrapped
        agent_pos = unwrapped.agent_pos
        agent_dir = unwrapped.agent_dir

        facing_dir_text = self.dir_map.get(agent_dir, str(agent_dir))
        carrying = "key" if unwrapped.carrying else "nothing"
        action_text = self.action_map.get(int(action), str(action))

        door_pos, door_state = None, "closed"
        key_pos = None
        goal_pos = None

        for i in range(unwrapped.grid.width):
            for j in range(unwrapped.grid.height):
                obj = unwrapped.grid.get(i, j)
                if obj is not None:
                    if obj.type == 'door':
                        door_pos = (i, j)
                        door_state = "open" if obj.is_open else "closed"
                    elif obj.type == 'key':
                        key_pos = (i, j)
                    elif obj.type == 'goal':
                        goal_pos = (i, j)

        fwd_pos = unwrapped.front_pos
        fwd_cell = unwrapped.grid.get(*fwd_pos)
        fwd_obj_type = fwd_cell.type if fwd_cell else "empty space"

        state_desc = (
            f"Agent at {tuple(agent_pos)}, facing {facing_dir_text}. "
            f"Forward vision: {fwd_obj_type}. "
            f"Inventory: {carrying}. "
            f"Door at {door_pos} is {door_state}. "
            f"Key at {key_pos}. Goal at {goal_pos}. "
            f"Action taken: {action_text}"
        )
        return state_desc

    def reset(self, **kwargs):
        self.visited_positions.clear()
        self.has_picked_up_key = False
        self.has_opened_door = False
        self.has_reached_goal = False
        obs, info = self.env.reset(**kwargs)
        self.visited_positions.add(tuple(self.env.unwrapped.agent_pos))
        return obs, info

    def step(self, action):
        prev_pos = tuple(self.env.unwrapped.agent_pos)
        state_text = self.get_text_description(action)

        obs, original_reward, terminated, truncated, info = self.env.step(action)

        deberta_reward = 0.0
        current_deberta_score = 0.0
        needs_evaluation = False

        if action == 2:
            if (tuple(self.env.unwrapped.agent_pos) == prev_pos and not terminated) or (terminated and original_reward > 0):
                needs_evaluation = True
        elif action in [3, 4, 5]:
            needs_evaluation = True

        if needs_evaluation:
            try:
                output = self.reward_pipeline(state_text)
                current_deberta_score = float(output[0]['label'])
            except Exception as e:
                current_deberta_score = 0.0

        if current_deberta_score < 0:
            deberta_reward += current_deberta_score
            if current_deberta_score <= -2.0:
                terminated = True

        elif current_deberta_score >= 1.0:
            reward_class = round(current_deberta_score)
            if reward_class == 2 and not self.has_picked_up_key:
                self.has_picked_up_key = True
                deberta_reward += current_deberta_score
            elif reward_class == 3 and not self.has_opened_door:
                self.has_opened_door = True
                deberta_reward += current_deberta_score
            elif reward_class == 10 and not self.has_reached_goal:
                self.has_reached_goal = True
                deberta_reward += current_deberta_score

        combined_reward = (original_reward * 10.0) + deberta_reward - 0.01

        return obs, combined_reward, terminated, truncated, info

print("DebertaRewardWrapper updated!")


In [ ]:
class MinigridFeaturesExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=128):
        super().__init__(observation_space, features_dim)
        n_input_channels = observation_space.shape[2]

        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )

        with th.no_grad():
            sample_tensor = th.as_tensor(observation_space.sample()[None]).float()
            permuted_sample = sample_tensor.permute(0, 3, 1, 2)
            n_flatten = self.cnn(permuted_sample).shape[1]

        self.linear = nn.Sequential(nn.Linear(n_flatten, features_dim), nn.ReLU())

    def forward(self, observations: th.Tensor) -> th.Tensor:
        permuted_observations = observations.permute(0, 3, 1, 2)
        return self.linear(self.cnn(permuted_observations))

In [ ]:
import os

log_dir = "./ppo_logs/"
os.makedirs(log_dir, exist_ok=True)

env_id = "MiniGrid-DoorKey-8x8-v0"

env = gym.make(env_id, render_mode="rgb_array", max_episode_steps=200)
env = DebertaRewardWrapper(env, reward_model_path="./reward-model-lora")
env = ImgObsWrapper(env)
env = Monitor(env, log_dir)

In [ ]:
from typing import Callable
import time

def linear_schedule(initial_value: float) -> Callable[[float], float]:
    """
    Linear learning rate schedule.
    """
    def func(progress_remaining: float) -> float:
        return progress_remaining * initial_value
    return func

policy_kwargs = dict(
    features_extractor_class=MinigridFeaturesExtractor,
    features_extractor_kwargs=dict(features_dim=128),
)

model = PPO(
    "CnnPolicy",
    env,
    policy_kwargs=policy_kwargs,
    verbose=1,
    tensorboard_log="./ppo_logs/",
    ent_coef=0.01,
    learning_rate=linear_schedule(3e-4), # Lowered for stability
    gamma=0.999, # Increased gamma for very long-horizon credit assignment
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    clip_range=0.2
)

print("Starting PPO training on 8x8 env...")
start_time = time.time()
model.learn(total_timesteps=1_000_000)
end_time = time.time()

print(f"\nPPO training complete in {end_time - start_time:.2f} seconds!")

env.close()

In [ ]:
import os
import glob

print("Checking TensorBoard event files in ./ppo_logs/:")
event_files = glob.glob('./ppo_logs/**/events.out.tfevents.*', recursive=True)
if event_files:
    for f in event_files:
        print(f"Found log: {f}")
    print("\nThe logs are successfully being written! Try clicking the refresh icon 🔄 in the top right of the TensorBoard UI.")
else:
    print("No TensorBoard logs found. Something might be wrong with the logging configuration.")

In [ ]:
eval_env = gym.make(env_id, render_mode="rgb_array", max_episode_steps=300) 
eval_env = ImgObsWrapper(eval_env)

In [ ]:
from stable_baselines3.common.evaluation import evaluate_policy

mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10)

print(f"Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

In [ ]:
def analyze_agent_behavior(model, env, num_episodes=50):
    print(f"Analyzing agent behavior over {num_episodes} episodes...")

    action_names = {0: 'turn left', 1: 'turn right', 2: 'move forward',
                    3: 'pick up', 4: 'drop', 5: 'toggle or open', 6: 'done'}

    total_pick_ups = 0
    total_drops = 0
    total_toggles = 0
    episodes_with_exploit = 0

    for ep in range(num_episodes):
        obs, _ = env.reset()
        done = False

        pick_ups_this_ep = 0
        drops_this_ep = 0
        toggles_this_ep = 0

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            act_val = int(action)
            if act_val == 3: pick_ups_this_ep += 1
            if act_val == 4: drops_this_ep += 1
            if act_val == 5: toggles_this_ep += 1

        if pick_ups_this_ep > 1 or toggles_this_ep > 1 or drops_this_ep > 0:
            episodes_with_exploit += 1

        total_pick_ups += pick_ups_this_ep
        total_drops += drops_this_ep
        total_toggles += toggles_this_ep

    print("-" * 40)
    print("BEHAVIOR STATISTICS:")
    print(f"Total 'pick up' actions: {total_pick_ups}")
    print(f"Total 'drop' actions: {total_drops}")
    print(f"Total 'toggle/open' actions: {total_toggles}")
    print(f"\nAverage pick ups per episode: {total_pick_ups / num_episodes:.2f} (Expected: ~1 or less)")
    print(f"Average drops per episode: {total_drops / num_episodes:.2f} (Expected: 0)")
    print(f"Average toggles per episode: {total_toggles / num_episodes:.2f} (Expected: ~1 or less)")

    print(f"\nEpisodes with exploit pattern detected (drop used, or >1 pickup/toggle): {episodes_with_exploit}/{num_episodes}")
    if episodes_with_exploit > 0:
        print("\n WARNING: The agent might be farming points! It is repeating one-time actions.")
    else:
        print("\n GOOD: The agent does not appear to be farming points via repetitive interactions.")

analyze_agent_behavior(model, eval_env, num_episodes=50)

In [ ]:
# Visualize 5 episodes in a single row
num_simulations = 5
fig, axes = plt.subplots(1, num_simulations, figsize=(20, 4))

for i in range(num_simulations):
    obs, info = eval_env.reset()
    rewards = 0
    agent_path = []
    agent_path.append(eval_env.unwrapped.agent_pos)

    for _ in range(200): 
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        rewards += reward
        agent_path.append(eval_env.unwrapped.agent_pos) 

        if terminated or truncated:
            break

    print(f"Episode {i+1} finished with reward: {rewards:.2f}")

    final_img = eval_env.render()

    ax = axes[i]
    ax.imshow(final_img)
    ax.set_title(f"Episode {i+1}\nReward: {rewards:.2f}")
    ax.axis('off')

    path_x = [p[0] for p in agent_path]
    path_y = [p[1] for p in agent_path]

    tile_size = 32
    pixel_x = [x * tile_size + tile_size / 2 for x in path_x]
    pixel_y = [y * tile_size + tile_size / 2 for y in path_y]

    ax.plot(pixel_x, pixel_y, color='red', linestyle='-', linewidth=1.5, marker='o', markersize=3, alpha=0.7)
    ax.scatter(pixel_x[0], pixel_y[0], color='green', marker='X', s=50, label="Start" if i==0 else "")
    ax.scatter(pixel_x[-1], pixel_y[-1], color='blue', marker='o', s=50, label="End" if i==0 else "")

axes[0].legend(loc='lower center', bbox_to_anchor=(0.5, -0.2), ncol=2)
plt.tight_layout()
plt.show()

eval_env.close()


In [ ]:
n_simulations = 100
max_steps_per_episode = 300

env_width = eval_env.unwrapped.width
env_height = eval_env.unwrapped.height

visit_counts = np.zeros((env_height, env_width), dtype=int)

print(f"Running {n_simulations} simulations...")

for i in range(n_simulations):
    obs, info = eval_env.reset()
    current_pos = eval_env.unwrapped.agent_pos
    visit_counts[current_pos[1], current_pos[0]] += 1

    for step in range(max_steps_per_episode):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)

        current_pos = eval_env.unwrapped.agent_pos
        visit_counts[current_pos[1], current_pos[0]] += 1

        if terminated or truncated:
            break

print("Simulations complete. Generating heatmap...")

In [ ]:
# Visualize the heatmap of visit counts
plt.figure(figsize=(8, 8))
sns.heatmap(visit_counts, cmap='hot_r', annot=False, fmt='d', linewidths=.5, linecolor='black')
plt.title(f'Agent Visit Heatmap over {n_simulations} Simulations')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')
# plt.gca().invert_yaxis()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob

log_file_path = "./ppo_logs/monitor.csv"

if not os.path.exists(log_file_path):
    print(f"Monitor log file '{log_file_path}' not found. Please ensure the environment was wrapped with `Monitor` and training was run.")
else:
    print(f"Found monitor file: {log_file_path}")

    df_monitor = pd.read_csv(log_file_path, skiprows=1)

    plt.figure(figsize=(12, 6))
    plt.plot(df_monitor['l'].cumsum(), df_monitor['r'], label='Episode Reward')

    # Calculate and plot a 50-episode rolling mean
    rolling_window = 50
    if len(df_monitor) >= rolling_window:
        rolling_mean_rewards = df_monitor['r'].rolling(window=rolling_window).mean()
        plt.plot(df_monitor['l'].cumsum(), rolling_mean_rewards, label=f'{rolling_window}-Episode Rolling Mean Reward', linestyle='--')

    plt.title('Episode Rewards Over Entire Training')
    plt.xlabel('Timesteps (Cumulative)')
    plt.ylabel('Reward')
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(12, 6))
    plt.plot(df_monitor['l'].cumsum(), df_monitor['l'], label='Episode Length')
    if len(df_monitor) >= rolling_window:
        rolling_mean_lengths = df_monitor['l'].rolling(window=rolling_window).mean()
        plt.plot(df_monitor['l'].cumsum(), rolling_mean_lengths, label=f'{rolling_window}-Episode Rolling Mean Length', linestyle='--')
    plt.title('Episode Lengths Over Entire Training')
    plt.xlabel('Timesteps (Cumulative)')
    plt.ylabel('Episode Length')
    plt.legend()
    plt.grid(True)
    plt.show()